<a href="https://colab.research.google.com/github/Faiq-danZ/worksafe-ai/blob/main/training/train_tabular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [11]:
def weighted_risk_loss(y_true, y_pred):
    # Mean Squared Error dasar
    mse = tf.square(y_true - y_pred)
    # Memberi bobot 2x lebih besar jika data aslinya adalah Risiko Tinggi (y=1)
    # Ini membantu model lebih sensitif terhadap potensi PHK
    weight = tf.where(y_true > 0.5, 2.0, 1.0)
    return tf.reduce_mean(mse * weight)

print("Custom Loss Function berhasil didefinisikan.")

Custom Loss Function berhasil didefinisikan.


In [12]:
def build_tabular_model(input_shape):
    # Menggunakan Input Layer (Ciri khas Functional API)
    inputs = tf.keras.Input(shape=(input_shape,))

    # Layer pertama dengan Batch Normalization
    x = layers.Dense(64, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    # Layer kedua
    x = layers.Dense(32, activation='relu')(x)

    # Output layer (Sigmoid menghasilkan skor antara 0 sampai 1)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    # Inisialisasi Model
    model = models.Model(inputs=inputs, outputs=outputs, name="WorkSafe_Tabular_Model")

    # Compile
    model.compile(
        optimizer='adam',
        loss=weighted_risk_loss,
        metrics=['mae', 'accuracy']
    )
    return model

# Inisialisasi model dengan asumsi 10 fitur input
model = build_tabular_model(input_shape=10)
model.summary()

Model: "WorkSafe_Tabular_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,073 (12.00 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 128 (512.00 B)

In [13]:
class RiskMonitoringCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Jika Mean Absolute Error (MAE) sudah di bawah 0.1, stop training
        if logs.get('mae') is not None and logs.get('mae') < 0.1:
            print(f"\n[INFO] Target MAE tercapai di epoch {epoch+1}. Menghentikan training!")
            self.model.stop_training = True

print("Custom Callback berhasil didefinisikan.")

Custom Callback berhasil didefinisikan.


In [14]:
# Buat data acak (100 baris, 10 kolom)
X_dummy = np.random.rand(100, 10)
y_dummy = np.random.randint(0, 2, size=(100, 1))

print("Memulai simulasi training...")
model.fit(
    X_dummy,
    y_dummy,
    epochs=15,
    batch_size=8,
    callbacks=[RiskMonitoringCallback()],
    verbose=1
)

Memulai simulasi training...
Epoch 1/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5100 - loss: 0.3782 - mae: 0.4972
Epoch 2/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5300 - loss: 0.3535 - mae: 0.4859 
Epoch 3/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5300 - loss: 0.3163 - mae: 0.4599 
Epoch 4/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5600 - loss: 0.3103 - mae: 0.4578 
Epoch 5/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5700 - loss: 0.3090 - mae: 0.4600 
Epoch 6/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6700 - loss: 0.2614 - mae: 0.4166 
Epoch 7/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6300 - loss: 0.2695 - mae: 0.4277 
Epoch 8/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6400 - loss: 0.2601 - mae: 0.4178 
Epoch 9/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6500 - loss: 0.2612 - mae: 0.4156 
Epoch 10/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6700 - loss: 0.2

In [16]:
save_path = 'models/tabular_model/'
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Simpan model dalam format .keras
model_file = os.path.join(save_path, 'worksafe_model_v1.keras')
model.save(model_file)

print(f"SUKSES! Model disimpan di: {model_file}")

SUKSES! Model disimpan di: models/tabular_model/worksafe_model_v1.keras
